In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from skimage.graph import pixel_graph
import json
from sklearn.cluster import DBSCAN

W, H, res = 200, 150, 0.1

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(30, 120, 40, 60); w(30, 120, 140, 160)
    w(10, 30, 10, 40); w(10, 30, 70, 100); w(10, 30, 120, 150)
    w(10, 30, 170, 200); w(10, 30, 220, 250)
    w(50, 70, 80, 120); w(50, 70, 180, 220)
    w(80, 100, 50, 90); w(80, 100, 130, 170); w(80, 100, 210, 250)
    w(120, 140, 70, 110); w(120, 140, 150, 190); w(120, 140, 230, 270)
    c(25, 25, 3); c(25, 175, 3); c(125, 25, 3); c(125, 175, 3)
    c(75, 75, 4); c(175, 75, 4); c(75, 125, 4); c(175, 125, 4)
    w(60, 80, 30, 50); w(60, 80, 190, 210)
    w(100, 120, 30, 50); w(100, 120, 190, 210)
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)
skel_bool = skeleton.astype(bool)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 使用 pixel_graph 构建图并提取端点/分叉点
# ==========================================
graph, flat_indices = pixel_graph(
    skel_bool, 
    connectivity=2, 
    edge_function=lambda x, y, d: d
)

degrees = np.diff(graph.indptr)
endpoint_indices = np.where(degrees == 1)[0]
junction_indices = np.where(degrees >= 3)[0]

coord_to_idx = {}
node_xy_coords = []

for i, flat_idx in enumerate(flat_indices):
    y, x = np.unravel_index(flat_idx, skel_bool.shape)
    coord_to_idx[(x, y)] = i
    node_xy_coords.append((y, x))

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 45 
MAX_STRAIGHT_STEPS = 15 

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue 
    
    n1, n2 = neis
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点，并建立 Graph Index
# ==========================================
raw_key_points = set()
for i in endpoint_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
for i in junction_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
raw_key_points.update(bend_points_set)

sampled_key_points = raw_key_points  # 暂时跳过采样

key_point_indices = set()
for x, y in sampled_key_points:
    if (x, y) in coord_to_idx:
        key_point_indices.add(coord_to_idx[(x, y)])

# ==========================================
# Step 5: 构造分叉路径
# ==========================================
def calc_local_span(x, y, s_set, radius=30):
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set: local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    return round(float((np.max(pts[:, 0]) - np.min(pts[:, 0])) * res), 2), \
           round(float((np.max(pts[:, 1]) - np.min(pts[:, 1])) * res), 2)

def trace_path(start_idx):
    neighbors = graph[start_idx].indices
    connected_nodes = []
    
    for n_idx in neighbors:
        if n_idx in key_point_indices:
            connected_nodes.append(n_idx)
        else:
            prev_idx = start_idx
            curr_idx = n_idx
            max_steps = 200
            steps = 0
            while degrees[curr_idx] == 2 and curr_idx not in key_point_indices and steps < max_steps:
                next_indices = graph[curr_idx].indices
                if len(next_indices) == 1:
                    break
                next_idx = next_indices[0] if next_indices[1] == prev_idx else next_indices[1]
                prev_idx, curr_idx = curr_idx, next_idx
                steps += 1
            
            if curr_idx in key_point_indices:
                connected_nodes.append(curr_idx)
            elif degrees[curr_idx] == 1:
                connected_nodes.append(curr_idx)
                
    return connected_nodes

edges_set = set()
for k_idx in key_point_indices:
    targets = trace_path(k_idx)
    for t_idx in targets:
        edge = tuple(sorted([k_idx, t_idx]))
        edges_set.add(edge)

# ==========================================
# Step 6: 节点合并优化 (修复版)
# ==========================================
def merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold=5):
    """
    简化版节点合并：直接遍历所有节点对，合并距离小于阈值的节点
    """
    current_indices = set(key_point_indices)
    merged_map = {}
    
    # 转换为 numpy 数组方便计算距离
    points = np.array([node_xy_coords[i] for i in current_indices])
    indices = list(current_indices)
    
    # 遍历所有节点对
    for i in range(len(indices)):
        for j in range(i+1, len(indices)):
            idx1, idx2 = indices[i], indices[j]
            
            # 检查两个节点是否都还在 current_indices 中
            if idx1 not in current_indices or idx2 not in current_indices:
                continue
                
            y1, x1 = node_xy_coords[idx1]
            y2, x2 = node_xy_coords[idx2]
            distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            if distance < merge_threshold:
                # 将 idx2 合并到 idx1
                merged_map[idx2] = idx1
                current_indices.remove(idx2)
    
    # 重新构建边集合
    final_edges = set()
    for edge in edges_set:
        # 替换被合并的节点
        if edge[0] in merged_map:
            edge = (merged_map[edge[0]], edge[1])
        if edge[1] in merged_map:
            edge = (edge[0], merged_map[edge[1]])
        
        # 确保边的两个端点都在当前节点集合中
        if edge[0] in current_indices and edge[1] in current_indices:
            final_edges.add(edge)
    
    return current_indices, final_edges

# 执行节点合并
merge_threshold = 5
# key_point_indices, edges_set = merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold)

# ==========================================
# 生成最终节点和边的 JSON 结构
# ==========================================
nodes = []
node_id_map_idx = {}
idx = 0
for k_idx in key_point_indices:
    y, x = node_xy_coords[k_idx]
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx,
        "span_y": sy
    })
    node_id_map_idx[k_idx] = node_id
    idx += 1

edges_json = []
for edge in edges_set:
    if edge[0] in node_id_map_idx and edge[1] in node_id_map_idx:
        edges_json.append({
            "from": node_id_map_idx[edge[0]],
            "to": node_id_map_idx[edge[1]]
        })

print(f"合并后提取到 {len(nodes)} 个纯导航节点 (含拐点)")
print(f"合并后提取到 {len(edges_json)} 条分叉/直线路径边")

# ==========================================
# 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    width_px = max(4, min(n['span_x'] * 10, 25))  
    height_px = max(4, min(n['span_y'] * 10, 25)) 
    
    ax.plot(px, py, 's', markersize=width_px, markeredgewidth=1.5, 
            color='red', markeredgecolor='black')
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pixel Graph Based Topology\n(Red=Key Nodes, Green=Traced Fork Paths)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pixel_graph_topology.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


合并后提取到 184 个纯导航节点 (含拐点)
合并后提取到 218 条分叉/直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 66.0,
      "y": 64.0,
      "span_x": 32.0,
      "span_y": 45.0
    },
    {
      "id": "N_1",
      "x": 193.0,
      "y": 139.0,
      "span_x": 30.0,
      "span_y": 34.0
    },
    {
      "id": "N_2",
      "x": 8.0,
      "y": 6.0,
      "span_x": 32.0,
      "span_y": 30.0
    },
    {
      "id": "N_3",
      "x": 132.0,
      "y": 64.0,
      "span_x": 32.0,
      "span_y": 60.0
    },
    {
      "id": "N_4",
      "x": 10.0,
      "y": 6.0,
      "span_x": 34.0,
      "span_y": 30.0
    },
    {
      "id": "N_5",
      "x": 193.0,
      "y": 140.0,
      "span_x": 30.0,
      "span_y": 33.0
    },
    {
      "id": "N_6",
      "x": 12.0,
      "y": 6.0,
      "span_x": 36.0,
      "span_y": 30.0
    },
    {
      "id": "N_7",
      "x": 11.0,
      "y": 6.0,
      "span_x": 35.0,
      "span_y": 30.0
    },
    {
      "id": "N_8",
      "x": 14.0,
      "y": 

In [39]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from skimage.graph import pixel_graph
import json
from sklearn.cluster import DBSCAN

W, H, res = 200, 150, 0.1

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(30, 120, 40, 60); w(30, 120, 140, 160)
    w(10, 30, 10, 40); w(10, 30, 70, 100); w(10, 30, 120, 150)
    w(10, 30, 170, 200); w(10, 30, 220, 250)
    w(50, 70, 80, 120); w(50, 70, 180, 220)
    w(80, 100, 50, 90); w(80, 100, 130, 170); w(80, 100, 210, 250)
    w(120, 140, 70, 110); w(120, 140, 150, 190); w(120, 140, 230, 270)
    c(25, 25, 3); c(25, 175, 3); c(125, 25, 3); c(125, 175, 3)
    c(75, 75, 4); c(175, 75, 4); c(75, 125, 4); c(175, 125, 4)
    w(60, 80, 30, 50); w(60, 80, 190, 210)
    w(100, 120, 30, 50); w(100, 120, 190, 210)
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)
skel_bool = skeleton.astype(bool)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 使用 pixel_graph 构建图并提取端点/分叉点
# ==========================================
graph, flat_indices = pixel_graph(
    skel_bool, 
    connectivity=2, 
    edge_function=lambda x, y, d: d
)

degrees = np.diff(graph.indptr)
endpoint_indices = np.where(degrees == 1)[0]
junction_indices = np.where(degrees >= 3)[0]

coord_to_idx = {}
node_xy_coords = []

for i, flat_idx in enumerate(flat_indices):
    y, x = np.unravel_index(flat_idx, skel_bool.shape)
    coord_to_idx[(x, y)] = i
    node_xy_coords.append((y, x))

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 45 
MAX_STRAIGHT_STEPS = 15 

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue 
    
    n1, n2 = neis
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点，并建立 Graph Index
# ==========================================
raw_key_points = set()
for i in endpoint_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
for i in junction_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
raw_key_points.update(bend_points_set)

sampled_key_points = raw_key_points  # 暂时跳过采样

key_point_indices = set()
for x, y in sampled_key_points:
    if (x, y) in coord_to_idx:
        key_point_indices.add(coord_to_idx[(x, y)])

# ==========================================
# Step 5: 构造分叉路径
# ==========================================
def calc_local_span(x, y, s_set, radius=30):
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set: local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    return round(float((np.max(pts[:, 0]) - np.min(pts[:, 0])) * res), 2), \
           round(float((np.max(pts[:, 1]) - np.min(pts[:, 1])) * res), 2)

def trace_path(start_idx):
    neighbors = graph[start_idx].indices
    connected_nodes = []
    
    for n_idx in neighbors:
        if n_idx in key_point_indices:
            connected_nodes.append(n_idx)
        else:
            prev_idx = start_idx
            curr_idx = n_idx
            max_steps = 200
            steps = 0
            while degrees[curr_idx] == 2 and curr_idx not in key_point_indices and steps < max_steps:
                next_indices = graph[curr_idx].indices
                if len(next_indices) == 1:
                    break
                next_idx = next_indices[0] if next_indices[1] == prev_idx else next_indices[1]
                prev_idx, curr_idx = curr_idx, next_idx
                steps += 1
            
            if curr_idx in key_point_indices:
                connected_nodes.append(curr_idx)
            elif degrees[curr_idx] == 1:
                connected_nodes.append(curr_idx)
                
    return connected_nodes

edges_set = set()
for k_idx in key_point_indices:
    targets = trace_path(k_idx)
    for t_idx in targets:
        edge = tuple(sorted([k_idx, t_idx]))
        edges_set.add(edge)

# ==========================================
# Step 6: 节点合并优化 (修复版)
# ==========================================
def merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold=5):
    """
    简化版节点合并：直接遍历所有节点对，合并距离小于阈值的节点
    """
    current_indices = set(key_point_indices)
    merged_map = {}
    
    # 转换为 numpy 数组方便计算距离
    points = np.array([node_xy_coords[i] for i in current_indices])
    indices = list(current_indices)
    
    # 遍历所有节点对
    for i in range(len(indices)):
        for j in range(i+1, len(indices)):
            idx1, idx2 = indices[i], indices[j]
            
            # 检查两个节点是否都还在 current_indices 中
            if idx1 not in current_indices or idx2 not in current_indices:
                continue
                
            y1, x1 = node_xy_coords[idx1]
            y2, x2 = node_xy_coords[idx2]
            distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            if distance < merge_threshold:
                # 将 idx2 合并到 idx1
                merged_map[idx2] = idx1
                current_indices.remove(idx2)
    
    # 重新构建边集合
    final_edges = set()
    for edge in edges_set:
        # 替换被合并的节点
        if edge[0] in merged_map:
            edge = (merged_map[edge[0]], edge[1])
        if edge[1] in merged_map:
            edge = (edge[0], merged_map[edge[1]])
        
        # 确保边的两个端点都在当前节点集合中
        if edge[0] in current_indices and edge[1] in current_indices:
            final_edges.add(edge)
    
    return current_indices, final_edges

# 执行节点合并
merge_threshold = 5
key_point_indices, edges_set = merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold)

# ==========================================
# 生成最终节点和边的 JSON 结构
# ==========================================
nodes = []
node_id_map_idx = {}
idx = 0
for k_idx in key_point_indices:
    y, x = node_xy_coords[k_idx]
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx,
        "span_y": sy
    })
    node_id_map_idx[k_idx] = node_id
    idx += 1

edges_json = []
for edge in edges_set:
    if edge[0] in node_id_map_idx and edge[1] in node_id_map_idx:
        edges_json.append({
            "from": node_id_map_idx[edge[0]],
            "to": node_id_map_idx[edge[1]]
        })

print(f"合并后提取到 {len(nodes)} 个纯导航节点 (含拐点)")
print(f"合并后提取到 {len(edges_json)} 条分叉/直线路径边")

# ==========================================
# 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # width_px = max(4, min(n['span_x'] * 10, 25))  
    # height_px = max(4, min(n['span_y'] * 10, 25)) 
    width_px = n['span_x'] * 10 
    height_px = n['span_y'] * 10
    
    ax.plot(px, py, 's', markersize=width_px, markeredgewidth=1.5, 
            color='red', markeredgecolor='black')
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pixel Graph Based Topology\n(Red=Key Nodes, Green=Traced Fork Paths)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pixel_graph_topology_merged.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


合并后提取到 47 个纯导航节点 (含拐点)
合并后提取到 91 条分叉/直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 6.6,
      "y": 6.4,
      "span_x": 3.2,
      "span_y": 4.5
    },
    {
      "id": "N_1",
      "x": 19.3,
      "y": 13.9,
      "span_x": 3.0,
      "span_y": 3.4
    },
    {
      "id": "N_2",
      "x": 0.8,
      "y": 0.6,
      "span_x": 3.2,
      "span_y": 3.0
    },
    {
      "id": "N_3",
      "x": 13.2,
      "y": 6.4,
      "span_x": 3.2,
      "span_y": 6.0
    },
    {
      "id": "N_4",
      "x": 1.4,
      "y": 0.6,
      "span_x": 3.8,
      "span_y": 3.0
    },
    {
      "id": "N_5",
      "x": 6.5,
      "y": 6.9,
      "span_x": 3.0,
      "span_y": 4.0
    },
    {
      "id": "N_6",
      "x": 19.4,
      "y": 0.5,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_7",
      "x": 13.4,
      "y": 6.9,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_8",
      "x": 6.6,
      "y": 7.4,
      "span_x": 3.1,
    

In [27]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from skimage.graph import pixel_graph
import json
from sklearn.cluster import DBSCAN

# W, H, res = 200, 150, 0.25

# def make_complex_grid():
#     g = np.ones((H, W), dtype=np.uint8) * 255
#     def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
#     def c(cy, cx, r):
#         Y, X = np.ogrid[:H, :W]
#         g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
#     w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
#     w(30, 120, 40, 60); w(30, 120, 140, 160)
#     w(10, 30, 10, 40); w(10, 30, 70, 100); w(10, 30, 120, 150)
#     w(10, 30, 170, 200); w(10, 30, 220, 250)
#     w(50, 70, 80, 120); w(50, 70, 180, 220)
#     w(80, 100, 50, 90); w(80, 100, 130, 170); w(80, 100, 210, 250)
#     w(120, 140, 70, 110); w(120, 140, 150, 190); w(120, 140, 230, 270)
#     c(25, 25, 3); c(25, 175, 3); c(125, 25, 3); c(125, 175, 3)
#     c(75, 75, 4); c(175, 75, 4); c(75, 125, 4); c(175, 125, 4)
#     w(60, 80, 30, 50); w(60, 80, 190, 210)
#     w(100, 120, 30, 50); w(100, 120, 190, 210)
#     return g

# grid = make_complex_grid()

W, H, res = 250, 200, 0.05  # ★ 放大画布、提高分辨率

def make_complex_grid_v2():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0

    # ============================================================
    # 1. 外墙边界
    # ============================================================
    w(0, 3, 0, W); w(H-3, H, 0, W)
    w(0, H, 0, 3); w(0, H, W-3, W)

    # ============================================================
    # 2. 主干道 —— 宽窄混合
    # ============================================================
    # 主横廊（宽）
    w(40, 65, 15, 235)
    # 主纵廊（宽）
    w(15, 185, 50, 75)
    # 次横廊（窄）
    w(110, 125, 15, 235)
    # 次纵廊（窄）
    w(65, 135, 120, 135)

    # ============================================================
    # 3. 对角通道（45° 方向）
    # ============================================================
    for i in range(50):
        # 左上 -> 右下 对角 1
        d = 3
        x = 80 + i
        y = 70 + i
        if 0 <= y < H and 0 <= x < W:
            w(y, y+d, x, x+d)
        # 右上 -> 左下 对角 2
        x2 = 200 - i
        y2 = 70 + i
        if 0 <= y2 < H and 0 <= x2 < W:
            w(y2, y2+d, x2-3, x2)
        # 短对角 次级
        x3 = 85 + i
        y3 = 140 + i
        if 0 <= y3 < H and 0 <= x3 < W:
            w(y3, y3+2, x3, x3+2)

    # ============================================================
    # 4. 不规则形状房间
    # ============================================================
    # L 形房间（左下）
    w(145, 175, 15, 45)
    w(160, 175, 15, 80)

    # T 形房间（右上）
    w(15, 40, 150, 180)
    w(15, 40, 175, 230)

    # Z 形走廊
    w(80, 95, 15, 45)
    w(95, 110, 30, 55)
    w(110, 125, 15, 40)

    # U 形房间（右下）
    w(145, 185, 155, 175)
    w(145, 185, 210, 230)
    w(165, 185, 155, 230)

    # 十字形房间（中下）
    w(155, 175, 85, 135)
    w(145, 185, 100, 120)

    # ============================================================
    # 5. 弧形弯道（用密集圆孔模拟弧线通道）
    # ============================================================
    center_y, center_x = 130, 230
    for angle_deg in range(200, 340):
        angle = np.radians(angle_deg)
        r = 18
        cx = int(center_x + r * np.cos(angle))
        cy = int(center_y + r * np.sin(angle))
        c(cy, cx, 4)

    # 第二段弧形（更大半径）
    center_y2, center_x2 = 175, 100
    for angle_deg in range(-90, 30):
        angle = np.radians(angle_deg)
        r = 22
        cx = int(center_x2 + r * np.cos(angle))
        cy = int(center_y2 + r * np.sin(angle))
        c(cy, cx, 3)

    # ============================================================
    # 6. 圆形障碍物 + 柱子
    # ============================================================
    # 大圆洞
    c(50, 100, 8)
    c(50, 160, 8)
    c(95, 165, 6)
    # 小柱子群
    for cy, cx in [(52, 130), (52, 140), (60, 130), (60, 140),
                    (120, 70), (120, 80), (130, 70), (130, 80),
                    (85, 195), (95, 195)]:
        c(cy, cx, 2)

    # ============================================================
    # 7. 菱形通道（通过旋转走廊近似）
    # ============================================================
    for i in range(35):
        t = i / 35.0 * np.pi * 0.5
        cx = int(160 + 25 * np.cos(t + np.pi/4))
        cy = int(85 + 25 * np.sin(t + np.pi/4))
        c(cy, cx, 3)

    # ============================================================
    # 8. 锥形走廊（逐渐变窄）
    # ============================================================
    for i in range(30):
        y = 25 + i
        half_w = max(2, 10 - i // 3)
        cx = 225
        w(y, y+1, cx - half_w, cx + half_w)

    # ============================================================
    # 9. 末端死胡同（各种朝向）
    # ============================================================
    # 朝上的
    w(140, 160, 40, 48)
    # 朝下的
    w(70, 80, 185, 193)
    # 朝左的
    w(42, 48, 60, 80)
    # 朝右的
    w(55, 61, 135, 150)

    # ============================================================
    # 10. 多通交叉路口
    # ============================================================
    # 五岔口
    c(125, 50, 5)
    w(125, 130, 35, 50)
    w(125, 130, 50, 65)

    return g


# ============================================================
# 运行完整流程（与之前完全一致）
# ============================================================
grid = make_complex_grid_v2()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)
skel_bool = skeleton.astype(bool)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 使用 pixel_graph 构建图并提取端点/分叉点
# ==========================================
graph, flat_indices = pixel_graph(
    skel_bool, 
    connectivity=2, 
    edge_function=lambda x, y, d: d
)

degrees = np.diff(graph.indptr)
endpoint_indices = np.where(degrees == 1)[0]
junction_indices = np.where(degrees >= 3)[0]

coord_to_idx = {}
node_xy_coords = []

for i, flat_idx in enumerate(flat_indices):
    y, x = np.unravel_index(flat_idx, skel_bool.shape)
    coord_to_idx[(x, y)] = i
    node_xy_coords.append((y, x))

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 20
MAX_STRAIGHT_STEPS = 10

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue 
    
    n1, n2 = neis
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点，并建立 Graph Index
# ==========================================
raw_key_points = set()
for i in endpoint_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
for i in junction_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
raw_key_points.update(bend_points_set)

sampled_key_points = raw_key_points  # 暂时跳过采样

key_point_indices = set()
for x, y in sampled_key_points:
    if (x, y) in coord_to_idx:
        key_point_indices.add(coord_to_idx[(x, y)])

# ==========================================
# Step 5: 构造分叉路径
# ==========================================
def calc_local_span(x, y, s_set, radius=30):
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set: local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    return round(float((np.max(pts[:, 0]) - np.min(pts[:, 0])) * res), 2), \
           round(float((np.max(pts[:, 1]) - np.min(pts[:, 1])) * res), 2)

def trace_path(start_idx):
    neighbors = graph[start_idx].indices
    connected_nodes = []
    
    for n_idx in neighbors:
        if n_idx in key_point_indices:
            connected_nodes.append(n_idx)
        else:
            prev_idx = start_idx
            curr_idx = n_idx
            max_steps = 200
            steps = 0
            while degrees[curr_idx] == 2 and curr_idx not in key_point_indices and steps < max_steps:
                next_indices = graph[curr_idx].indices
                if len(next_indices) == 1:
                    break
                next_idx = next_indices[0] if next_indices[1] == prev_idx else next_indices[1]
                prev_idx, curr_idx = curr_idx, next_idx
                steps += 1
            
            if curr_idx in key_point_indices:
                connected_nodes.append(curr_idx)
            elif degrees[curr_idx] == 1:
                connected_nodes.append(curr_idx)
                
    return connected_nodes

edges_set = set()
for k_idx in key_point_indices:
    targets = trace_path(k_idx)
    for t_idx in targets:
        edge = tuple(sorted([k_idx, t_idx]))
        edges_set.add(edge)

# ==========================================
# Step 6: 节点合并优化 (修复版)
# ==========================================
def merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold=5):
    """
    简化版节点合并：直接遍历所有节点对，合并距离小于阈值的节点
    """
    current_indices = set(key_point_indices)
    merged_map = {}
    
    # 转换为 numpy 数组方便计算距离
    points = np.array([node_xy_coords[i] for i in current_indices])
    indices = list(current_indices)
    
    # 遍历所有节点对
    for i in range(len(indices)):
        for j in range(i+1, len(indices)):
            idx1, idx2 = indices[i], indices[j]
            
            # 检查两个节点是否都还在 current_indices 中
            if idx1 not in current_indices or idx2 not in current_indices:
                continue
                
            y1, x1 = node_xy_coords[idx1]
            y2, x2 = node_xy_coords[idx2]
            distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            if distance < merge_threshold:
                # 将 idx2 合并到 idx1
                merged_map[idx2] = idx1
                current_indices.remove(idx2)
    
    # 重新构建边集合
    final_edges = set()
    for edge in edges_set:
        # 替换被合并的节点
        if edge[0] in merged_map:
            edge = (merged_map[edge[0]], edge[1])
        if edge[1] in merged_map:
            edge = (edge[0], merged_map[edge[1]])
        
        # 确保边的两个端点都在当前节点集合中
        if edge[0] in current_indices and edge[1] in current_indices:
            final_edges.add(edge)
    
    return current_indices, final_edges

# 执行节点合并
merge_threshold = 5
key_point_indices, edges_set = merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold)

# ==========================================
# 生成最终节点和边的 JSON 结构
# ==========================================
nodes = []
node_id_map_idx = {}
idx = 0
for k_idx in key_point_indices:
    y, x = node_xy_coords[k_idx]
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx,
        "span_y": sy
    })
    node_id_map_idx[k_idx] = node_id
    idx += 1

edges_json = []
for edge in edges_set:
    if edge[0] in node_id_map_idx and edge[1] in node_id_map_idx:
        edges_json.append({
            "from": node_id_map_idx[edge[0]],
            "to": node_id_map_idx[edge[1]]
        })

print(f"合并后提取到 {len(nodes)} 个纯导航节点 (含拐点)")
print(f"合并后提取到 {len(edges_json)} 条分叉/直线路径边")

# ==========================================
# 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

import matplotlib.patches as patches

# ... 前面的代码不变 ...

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    
    # 注意：这里假设你想在 数据坐标系 下体现真实的长宽比例
    # 如果你依然想保持在屏幕上固定视觉大小(点坐标)，需要做坐标转换
    # 这里提供两种思路：

    # ================= 思路一：在数据坐标系中画真实比例（推荐用于拓扑图） =================
    # 直接将 span_x, span_y 作为数据坐标的宽高
    w_data = n['span_x']
    h_data = n['span_y']
    
    # 创建矩形，注意 (px, py) 是中心点，Rectangle 的 xy 参数是左下角
    rect = patches.Rectangle(
        (px - w_data/2, py - h_data/2),  # 左下角坐标
        w_data,                          # 宽度
        h_data,                          # 高度
        linewidth=1.5, 
        edgecolor='black', 
        facecolor='red', 
        alpha=0.8
    )
    ax.add_patch(rect)
    
    # # 标注
    # ax.annotate(f"{n['span_x']:.1f}x{n['span_y']:.1f}", (px, py),
    #             textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')
    # 只添加节点ID指标
    ax.annotate(n['id'], (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

    # ================= 思路二：在屏幕显示坐标系中保持你原来的大小限制逻辑 =================
    # 如果你的 span 是归一化的极小值（如0.05），直接画在数据坐标里会看不见，
    # 你必须将其转换为 Display 坐标（点 points），但这在 matplotlib 中非常繁琐，
    # 通常需要借助 inset_axes 或复杂的 transform。

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pixel Graph Based Topology\n(Red=Key Nodes, Green=Traced Fork Paths)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pixel_graph_topology_merged.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


合并后提取到 176 个纯导航节点 (含拐点)
合并后提取到 333 条分叉/直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 2.2,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.95
    },
    {
      "id": "N_1",
      "x": 2.45,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.7
    },
    {
      "id": "N_2",
      "x": 3.7,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_3",
      "x": 3.95,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_4",
      "x": 7.2,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_5",
      "x": 7.45,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_6",
      "x": 11.4,
      "y": 0.4,
      "span_x": 2.1,
      "span_y": 1.5
    },
    {
      "id": "N_7",
      "x": 11.65,
      "y": 0.4,
      "span_x": 1.85,
      "span_y": 1.5
    },
    {
      "id": "N_8",
      "x": 4.15,
      "y": 0.55,
      "span_x"

In [45]:
import math
import heapq

# ==========================================
# 1. 构建带权重的邻接表
# ==========================================
def build_graph(nodes, edges):
    """
    将提取的 nodes 和 edges 转换为图算法需要的邻接表格式
    """
    graph = {}
    node_pos = {n['id']: (n['x'], n['y']) for n in nodes}
    
    # 初始化每个节点的邻居列表
    for n in nodes:
        graph[n['id']] = []
        
    # 计算边的欧几里得距离作为权重
    for e in edges:
        n1, n2 = e['from'], e['to']
        x1, y1 = node_pos[n1]
        x2, y2 = node_pos[n2]
        distance = math.hypot(x2 - x1, y2 - y1) # 等同于 sqrt((x2-x1)**2 + (y2-y1)**2)
        
        # 无向图，双向添加
        graph[n1].append((n2, distance))
        graph[n2].append((n1, distance))
        
    return graph, node_pos

# ==========================================
# 2. 核心 A* 寻路算法
# ==========================================
def astar_algorithm(graph, node_pos, start_id, end_id):
    """
    使用 A* 算法寻找最短路径
    返回: (路径节点列表, 总物理距离)
    """
    if start_id not in graph or end_id not in graph:
        return None, 0
        
    if start_id == end_id:
        return [start_id], 0.0

    # 优先队列 (f_cost, counter, node_id)
    # counter 用于打破 f_cost 相同时的平局，避免比较 tuple 报错
    open_list = [(0, 0, start_id)]
    counter = 1 
    
    # 记录从起点到当前节点的实际最短距离 g_cost
    g_costs = {start_id: 0.0}
    
    # 记录路径的父节点
    parents = {start_id: None}
    
    # 已访问节点集合
    closed_list = set()
    
    # 启发式函数：两点之间的直线距离
    def heuristic(node_id):
        x1, y1 = node_pos[node_id]
        x2, y2 = node_pos[end_id]
        return math.hypot(x2 - x1, y2 - y1)

    while open_list:
        current_f, _, current_id = heapq.heappop(open_list)
        
        # 找到终点，回溯路径
        if current_id == end_id:
            path = []
            while current_id is not None:
                path.append(current_id)
                current_id = parents[current_id]
            return path[::-1], g_costs[end_id] # 反转路径，返回总距离
            
        if current_id in closed_list:
            continue
        closed_list.add(current_id)
        
        # 遍历相邻节点
        for neighbor_id, edge_dist in graph[current_id]:
            if neighbor_id in closed_list:
                continue
                
            # 计算新的 g_cost
            tentative_g = g_costs[current_id] + edge_dist
            
            # 如果发现更短的路径，或者该节点还没被记录过
            if tentative_g < g_costs.get(neighbor_id, float('inf')):
                parents[neighbor_id] = current_id
                g_costs[neighbor_id] = tentative_g
                f_cost = tentative_g + heuristic(neighbor_id)
                heapq.heappush(open_list, (f_cost, counter, neighbor_id))
                counter += 1
                
    return None, 0.0 # 没找到路径

# ==========================================
# 3. 测试与运行
# ==========================================
# 假设前面已经提取好了 nodes 和 edges_json
graph, node_pos = build_graph(nodes, edges_json)

# 随便挑两个节点测试 (比如第 0 个节点 和 第 50 个节点，如果有的话)
# 实际使用时，这里的 start_id 和 end_id 由大模型解析自然语言后提供
start_node = nodes[100]['id']
end_node = nodes[90]['id'] 

print(f"\n开始寻路: {start_node} -> {end_node}")
path, total_dist = astar_algorithm(graph, node_pos, start_node, end_node)

if path:
    print(f"✅ 找到最短路径，经过 {len(path)} 个节点，总物理距离: {total_dist:.2f} 米")
    print(f"路径节点序列: {' -> '.join(path)}")
else:
    print("❌ 两点之间不可达")



开始寻路: N_100 -> N_90
✅ 找到最短路径，经过 78 个节点，总物理距离: 42.53 米
路径节点序列: N_100 -> N_101 -> N_105 -> N_110 -> N_125 -> N_122 -> N_133 -> N_128 -> N_121 -> N_120 -> N_119 -> N_118 -> N_132 -> N_140 -> N_151 -> N_161 -> N_168 -> N_175 -> N_174 -> N_173 -> N_167 -> N_163 -> N_159 -> N_162 -> N_166 -> N_165 -> N_164 -> N_158 -> N_157 -> N_156 -> N_155 -> N_150 -> N_147 -> N_136 -> N_130 -> N_111 -> N_106 -> N_102 -> N_92 -> N_81 -> N_46 -> N_34 -> N_24 -> N_22 -> N_21 -> N_20 -> N_14 -> N_15 -> N_11 -> N_0 -> N_1 -> N_2 -> N_3 -> N_8 -> N_12 -> N_16 -> N_17 -> N_18 -> N_19 -> N_13 -> N_4 -> N_5 -> N_6 -> N_7 -> N_9 -> N_10 -> N_23 -> N_26 -> N_35 -> N_43 -> N_65 -> N_64 -> N_69 -> N_68 -> N_77 -> N_76 -> N_80 -> N_90


In [44]:
map_diagram = {"nodes": nodes, "edges": edges_json}

In [2]:
import zai
print(zai.__version__)

0.2.2


In [42]:
[
        {
            "role": "system",
            "content": f"""
            你是一个拓扑图路线规划专家。请根据用户给出的拓扑图D以及起始节点Ns和目标节点Ne，给出对应的路线。
            D为json格式，'nodes'键中包含了所有的节点，名称以'N_i'的格式命名，'edges'包含了边，格式为'from'某个节点'to'某个节点，不需要考虑边方向。
            """
        },
        {
            "role": "user",
            "content": f" D: {map_diagram}; Ns: N_100; Ne: N_90;"
        }
    ]

[{'role': 'system',
  'content': "\n            你是一个拓扑图路线规划专家。请根据用户给出的拓扑图D以及起始节点Ns和目标节点Ne，给出对应的路线。\n            D为json格式，'nodes'键中包含了所有的节点，名称以'N_i'的格式命名，'edges'包含了边，格式为'from'某个节点'to'某个节点，不需要考虑边方向。\n            "},
 {'role': 'user',
  'content': " D: {'nodes': [{'id': 'N_0', 'x': 2.2, 'y': 0.4, 'span_x': 3.0, 'span_y': 0.95}, {'id': 'N_1', 'x': 2.45, 'y': 0.4, 'span_x': 3.0, 'span_y': 0.7}, {'id': 'N_2', 'x': 3.7, 'y': 0.4, 'span_x': 3.0, 'span_y': 0.65}, {'id': 'N_3', 'x': 3.95, 'y': 0.4, 'span_x': 3.0, 'span_y': 0.65}, {'id': 'N_4', 'x': 7.2, 'y': 0.4, 'span_x': 3.0, 'span_y': 0.65}, {'id': 'N_5', 'x': 7.45, 'y': 0.4, 'span_x': 3.0, 'span_y': 0.65}, {'id': 'N_6', 'x': 11.4, 'y': 0.4, 'span_x': 2.1, 'span_y': 1.5}, {'id': 'N_7', 'x': 11.65, 'y': 0.4, 'span_x': 1.85, 'span_y': 1.5}, {'id': 'N_8', 'x': 4.15, 'y': 0.55, 'span_x': 3.0, 'span_y': 0.65}, {'id': 'N_9', 'x': 11.9, 'y': 0.55, 'span_x': 1.6, 'span_y': 1.65}, {'id': 'N_10', 'x': 11.9, 'y': 0.8, 'span_x': 1.6, 'span_y': 1.9}, {'id

In [40]:
from zai import ZhipuAiClient
import json

# 初始化客户端
client = ZhipuAiClient(api_key="d2ce696b586b4657a6ec77005f26dd82.zPMQ531oxnTYfs5v")

# 基础 JSON 模式
response = client.chat.completions.create(
    model="GLM-4-Flash-250414",
    messages=[
        {
            "role": "system",
            "content": f"""
            你是一个拓扑图路线规划专家。请根据用户给出的拓扑图D以及起始节点Ns和目标节点Ne，给出对应的路线。
            D为json格式，'nodes'键中包含了所有的节点，名称以'N_i'的格式命名，'edges'包含了边，格式为'from'某个节点'to'某个节点，不需要考虑边方向。
            """
        },
        {
            "role": "user",
            "content": f" D: {map_diagram}; Ns: N_100; Ne: N_90;"
        }
    ],
    # response_format={
    #     "type": "json_object"
    # }
)

# 解析结果
result = json.loads(response.choices[0].message.content)
print(f"情感: {result['sentiment']}")
print(f"置信度: {result['confidence']}")
print(f"情绪: {result['emotions']}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [41]:
response

Completion(model='GLM-4-Flash-250414', created=1776173440, choices=[CompletionChoice(index=0, finish_reason='stop', message=CompletionMessage(content='为了找到从节点N_100到节点N_90的最短路径，我们可以使用图搜索算法，比如Dijkstra算法。但是由于您没有提供具体的距离或权重信息，我们将简化问题，假设每条边的权重相同，并使用BFS（广度优先搜索）来找到一条路径。\n\n下面是使用BFS找到路径的步骤：\n\n1. 初始化一个队列，将起始节点N_100加入队列。\n2. 初始化一个集合，用于记录已访问的节点，并将N_100加入该集合。\n3. 当队列不为空时，执行以下步骤：\n   a. 从队列中取出一个节点，记为当前节点。\n   b. 检查当前节点是否为目标节点N_90，如果是，则返回路径。\n   c. 遍历当前节点的所有邻居节点，对于每个邻居节点：\n      i. 如果邻居节点不在已访问集合中，则将邻居节点加入队列，并将其加入已访问集合。\n      ii. 记录邻居节点的前驱节点为当前节点。\n4. 如果队列为空且未找到目标节点，则返回无路径。\n\n由于我无法实际执行代码，我将提供伪代码来描述上述算法：\n\n```\nqueue = [Ns]\nvisited = set([Ns])\npredecessors = {Ns: None}\n\nwhile queue:\n    current_node = queue.pop(0)\n    if current_node == Ne:\n        path = []\n        while current_node is not None:\n            path.append(current_node)\n            current_node = predecessors[current_node]\n        return path[::-1]\n\n    for neighbor in get_neighbors(current_node):\n        if neighbor no

In [48]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

class RobotState(TypedDict):
    user_command: str        # 用户指令："去厨房把桌上的红苹果拿给我"
    topology_graph: dict     # 最新的拓扑图 JSON (带节点物品属性)
    start_node: str          # LLM 解析出的起点
    target_node: str         # LLM 解析出的终点
    target_object: str       # LLM 解析出的目标物体
    macro_path: List[str]    # A* 算出的拓扑节点序列
    current_nav_status: str  # 导航状态反馈 ("arrived_at_node", "blocked")

# 1. 意图解析节点 (LLM)
def parse_intent(state: RobotState):
    # Prompt: "根据拓扑图节点绑定的物体，找到起点和终点节点"
    # 输入: user_command + topology_graph
    # 输出: start_node, target_node, target_object
    return {"start_node": "N_5", "target_node": "N_42", "target_object": "红苹果"}

# 2. 宏观寻路节点 (图算法)
def plan_macro_path(state: RobotState):
    # 调用你刚才写的 astar_algorithm
    path, dist = astar_algorithm(graph, node_pos, state['start_node'], state['target_node'])
    return {"macro_path": path}

# 3. 任务下发节点 (接口调用)
def dispatch_navigation(state: RobotState):
    # 将 macro_path 发给底层 ROS2 导航栈 (比如依次发送目标点给 Nav2)
    # 这里是异步等待，底层走到节点后会反馈状态
    return {"current_nav_status": "moving"}

# 4. 状态监控与重规划节点 (动态调整)
def monitor_and_react(state: RobotState):
    status = state['current_nav_status']
    if status == "arrived_at_target_node":
        # 到达终点，触发抓取/其他动作
        return END
    elif status == "blocked":
        # 比如 N_15 走不通（门关了），触发重规划或让 LLM 重新决策
        return "parse_intent" # 跳回 LLM
    else:
        # 继续等待
        return "monitor_and_react"

# 构建图
workflow = StateGraph(RobotState)
workflow.add_node("parse_intent", parse_intent)
workflow.add_node("plan_path", plan_macro_path)
workflow.add_node("dispatch_nav", dispatch_navigation)
workflow.add_node("monitor", monitor_and_react)

workflow.set_entry_point("parse_intent")
workflow.add_edge("parse_intent", "plan_path")
workflow.add_edge("plan_path", "dispatch_nav")
workflow.add_edge("dispatch_nav", "monitor")

app = workflow.compile()

# ✨ 核心：画流程图！
app.get_graph().draw_mermaid_png(output_file_path="./workflow.png")

b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x8c\x00\x00\x02\x13\x08\x02\x00\x00\x00Z\rc\xd6\x00\x00\x10\x00IDATx\x9c\xec\x9d\x07@\x14\xc7\x1a\xc7gw\xef\x80\xa3\x8a\x95*`E,\x80=1\xb1a\x8b\xbd\xc5\xae\xb1%\x9ah\x8c1\xf6\x96X\x12\xbb1\xc6\x9ef\x7f\xc6\x18[\x8c\xb1Dc\xc3\xa8\x18{\x17\x04D\x05Tz\xbf\xb2\xfb\xbe\xbd\x85\xe3\xc0\x83\xe3\xb8\xdd\xe3v\x99_|\xbc\xbd\xd9\xd9\xd9\xf2\xdf\x99\xf9\xa6\xec72\x86a\x10\xc6\xba\x91!\x8c\xd5\x83E\x12\x01X$\x11\x80E\x12\x01X$\x11\x80E\x12\x01e R\xf8\xed\x94\'72\x93^\xaa4jF\x93C\xd3\x04!\x93\x11j5C\xc9H\x8d\x9a&I\x82aA29\xa9V\xd1\x10\x9f \xd9?\x04\x81h\r\xdbZ\xa0d\x04\x1c\xc8nP\x84F\x1b\x02\xbb\xe0\x7f\x0c\xcd\xc0\xb1\x10\x99\xdb\xcb\xde\x9b6Yv7\xc3\xe8\x8e\xe2\x80\x984\x9d\xff\x93 \xd9\xc3\xb9\xd3\xc0\xb9)9\xa9\xd1\x9eZ\x87\xdc\x96\x90\xc9\t\x85\xa3\xcc\xbb\xaemP\xebJ\xc8\xb2\x10\x16k\']?\x93p\xf3\\Zf\xaa\x1aNH\xc9\x90\xdc\x86\xb4\xb1#\x11\r\x0f\x90 e\x88V#\xee/C \x10\x85\x8d#\'4*\xb86\x86\xd0\xca\xc6>V\r\x9bNn46\x11\x82\x93\r\x11Z!h\xf6XP\x8e\xce\x